<a href="https://colab.research.google.com/github/herrypu/HP_LLM4CD_S6SP/blob/main/chipchat_exampleB_i3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM4ChipDesign HW1 Chipchat
## Example B (Example 2: Sequence Detector)


In [1]:
API_KEY="sk-4b4b50b8153147759b5530268e51a1f7"
MAX_TOKENS=4068
TEMPERATURE=0.7

### 1. Setting up

In [2]:
### Installing dependencies
!pip install openai

!apt-get update
!apt-get install -y iverilog

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
iverilog is already the newest version (11.0-1.1).
0 upgraded, 0 newly installed, 0 to remove 

In [3]:
def ask_deepseek(prompt):
    from openai import OpenAI
    client = OpenAI(
        api_key = API_KEY,
        base_url = "https://api.deepseek.com"
    )
    completion = client.chat.completions.create(
        model="deepseek-reasoner",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE
    )
    content = completion.choices[0].message.content

    print("-" * 30 + " LLM Response " + "-" * 30)
    print(content)
    print("-" * 74)

    return content

### 2. Verilog Generation

In [4]:
! mkdir -p sequence_detector
! cd sequence_detector && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/sequence_detector_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2114  100  2114    0     0  10746      0 --:--:-- --:--:-- --:--:-- 10785


In [5]:
verilog_generation_prompt = '''
I am trying to create a Verilog model named "sequence_detector".
It must meet the following hardware specifications to pass the testbench:

    - Port Names (Strict):
        - Clock: "clk"
        - Reset: "reset_n" (Active-low reset)
        - Input Data: "data" (3 bits)
        - Output: "sequence_found" (1 bit)

    - Detection Sequence:
        While enabled, it should detect the following 8-step sequence:
        1. 0b001
        2. 0b101
        3. 0b110
        4. 0b000
        5. 0b110
        6. 0b110
        7. 0b011
        8. 0b101

    - Requirements:
        - The "sequence_found" output should be high for one clock cycle immediately after the 8th value is detected.
        - The design must be synthesizable RTL.
'''

In [6]:
ask_deepseek(verilog_generation_prompt)

------------------------------ LLM Response ------------------------------
Here's a Verilog implementation of the sequence detector that meets your specifications:

```verilog
module sequence_detector (
    input wire clk,
    input wire reset_n,
    input wire [2:0] data,
    output reg sequence_found
);

    // Define states for FSM
    typedef enum logic [3:0] {
        IDLE = 4'd0,
        S1   = 4'd1,  // Expect 001
        S2   = 4'd2,  // Expect 101
        S3   = 4'd3,  // Expect 110
        S4   = 4'd4,  // Expect 000
        S5   = 4'd5,  // Expect 110
        S6   = 4'd6,  // Expect 110
        S7   = 4'd7,  // Expect 011
        S8   = 4'd8,  // Expect 101
        MATCH = 4'd9   // Sequence completed
    } state_t;

    // State registers
    state_t current_state, next_state;

    // Sequential logic for state transitions
    always_ff @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            current_state <= IDLE;
            sequence_found <= 1'b0;


"Here's a Verilog implementation of the sequence detector that meets your specifications:\n\n```verilog\nmodule sequence_detector (\n    input wire clk,\n    input wire reset_n,\n    input wire [2:0] data,\n    output reg sequence_found\n);\n\n    // Define states for FSM\n    typedef enum logic [3:0] {\n        IDLE = 4'd0,\n        S1   = 4'd1,  // Expect 001\n        S2   = 4'd2,  // Expect 101\n        S3   = 4'd3,  // Expect 110\n        S4   = 4'd4,  // Expect 000\n        S5   = 4'd5,  // Expect 110\n        S6   = 4'd6,  // Expect 110\n        S7   = 4'd7,  // Expect 011\n        S8   = 4'd8,  // Expect 101\n        MATCH = 4'd9   // Sequence completed\n    } state_t;\n\n    // State registers\n    state_t current_state, next_state;\n\n    // Sequential logic for state transitions\n    always_ff @(posedge clk or negedge reset_n) begin\n        if (!reset_n) begin\n            current_state <= IDLE;\n            sequence_found <= 1'b0;\n        end else begin\n            curren

### 3. Extraction and Verification

In [9]:
output_verilog_code = '''
module sequence_detector (
    input wire clk,
    input wire reset_n,
    input wire [2:0] data,
    output reg sequence_found
);

    // Define states for FSM
    typedef enum logic [3:0] {
        IDLE = 4'd0,
        S1   = 4'd1,  // Expect 001
        S2   = 4'd2,  // Expect 101
        S3   = 4'd3,  // Expect 110
        S4   = 4'd4,  // Expect 000
        S5   = 4'd5,  // Expect 110
        S6   = 4'd6,  // Expect 110
        S7   = 4'd7,  // Expect 011
        S8   = 4'd8,  // Expect 101
        MATCH = 4'd9   // Sequence completed
    } state_t;

    // State registers
    state_t current_state, next_state;

    // Sequential logic for state transitions
    always_ff @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            current_state <= IDLE;
            sequence_found <= 1'b0;
        end else begin
            current_state <= next_state;

            // Output is high for one cycle when sequence is detected
            sequence_found <= (next_state == MATCH) ? 1'b1 : 1'b0;
        end
    end

    // Combinational logic for next state and output
    always_comb begin
        // Default assignments
        next_state = current_state;

        case (current_state)
            IDLE: begin
                // Look for first sequence element (001)
                if (data == 3'b001) begin
                    next_state = S1;
                end
            end

            S1: begin
                // Expect second element (101)
                if (data == 3'b101) begin
                    next_state = S2;
                end else if (data == 3'b001) begin
                    // Restart detection if first element appears again
                    next_state = S1;
                end else begin
                    next_state = IDLE;
                end
            end

            S2: begin
                // Expect third element (110)
                if (data == 3'b110) begin
                    next_state = S3;
                end else if (data == 3'b001) begin
                    next_state = S1;
                end else begin
                    next_state = IDLE;
                end
            end

            S3: begin
                // Expect fourth element (000)
                if (data == 3'b000) begin
                    next_state = S4;
                end else if (data == 3'b001) begin
                    next_state = S1;
                end else begin
                    next_state = IDLE;
                end
            end

            S4: begin
                // Expect fifth element (110)
                if (data == 3'b110) begin
                    next_state = S5;
                end else if (data == 3'b001) begin
                    next_state = S1;
                end else begin
                    next_state = IDLE;
                end
            end

            S5: begin
                // Expect sixth element (110)
                if (data == 3'b110) begin
                    next_state = S6;
                end else if (data == 3'b001) begin
                    next_state = S1;
                end else begin
                    next_state = IDLE;
                end
            end

            S6: begin
                // Expect seventh element (011)
                if (data == 3'b011) begin
                    next_state = S7;
                end else if (data == 3'b001) begin
                    next_state = S1;
                end else begin
                    next_state = IDLE;
                end
            end

            S7: begin
                // Expect eighth element (101)
                if (data == 3'b101) begin
                    next_state = MATCH;
                end else if (data == 3'b001) begin
                    next_state = S1;
                end else begin
                    next_state = IDLE;
                end
            end

            MATCH: begin
                // After detection, look for next sequence
                if (data == 3'b001) begin
                    next_state = S1;
                end else begin
                    next_state = IDLE;
                end
            end

            default: begin
                next_state = IDLE;
            end
        endcase
    end

endmodule
'''
filename = "sequence_detector/sequence_detector.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [11]:
!cd sequence_detector/ && iverilog -g2012 -o sequence_detector.vvp sequence_detector.v sequence_detector_tb.v && vvp sequence_detector.vvp

Error: Cycle 8, Expected: 1, Got: 0
